In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import joblib

In [2]:
df = pd.read_csv("../data/processed/atm_model_ready.csv")

df.head()

,ATM Name,Transaction Date,No Of Withdrawals,No Of XYZ Card Withdrawals,No Of Other Card Withdrawals,Total amount Withdrawn,Amount withdrawn XYZ Card,Amount withdrawn Other Card,Weekday,Festival Religion,...,Is_Weekend,Week_of_Year,Demand_Outlier,Lag_1_Day,Lag_7_Day,Lag_14_Day,Lag_30_Day,Rolling_7_Day_Mean,Rolling_14_Day_Mean,Rolling_30_Day_Mean
0,Airport ATM,2011-01-01,98,56,42,10772.76,7440.78,3331.98,Saturday,C,...,1,0,0,10772.76,10772.76,10772.76,10772.76,10772.760000,10772.760000,10772.760000
1,Airport ATM,2011-01-02,105,78,27,12371.34,9388.18,2983.16,Tuesday,NH,...,1,1,0,10772.76,10772.76,10772.76,10772.76,10772.760000,10772.760000,10772.760000
2,Airport ATM,2011-01-03,119,101,18,11108.74,9955.28,1153.46,Tuesday,NH,...,0,1,0,12371.34,10772.76,10772.76,10772.76,11572.050000,11572.050000,11572.050000
3,Airport ATM,2011-01-04,96,80,16,10286.98,9109.98,1177.00,Friday,NH,...,0,1,0,11108.74,10772.76,10772.76,10772.76,11417.613333,11417.613333,11417.613333
4,Airport ATM,2011-01-05,131,63,68,10034.46,6567.66,3466.80,Sunday,N,...,0,1,0,10286.98,10772.76,10772.76,10772.76,11134.955000,11134.955000,11134.955000


In [3]:
print("Shape:", df.shape)
print(df.isnull().sum())

Shape: (11589, 26)
ATM Name                        0
Transaction Date                0
No Of Withdrawals               0
No Of XYZ Card Withdrawals      0
No Of Other Card Withdrawals    0
Total amount Withdrawn          0
Amount withdrawn XYZ Card       0
Amount withdrawn Other Card     0
Weekday                         0
Festival Religion               0
Working Day                     0
Holiday Sequence                0
Year                            0
Month                           0
Day                             0
Day_of_Week                     0
Is_Weekend                      0
Week_of_Year                    0
Demand_Outlier                  0
Lag_1_Day                       0
Lag_7_Day                       0
Lag_14_Day                      0
Lag_30_Day                      0
Rolling_7_Day_Mean              0
Rolling_14_Day_Mean             0
Rolling_30_Day_Mean             0
dtype: int64


In [4]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [5]:
target = "Total amount Withdrawn"

In [6]:
df = df.sort_values("Transaction Date").reset_index(drop=True)

In [7]:
print("Start:", df["Transaction Date"].min())
print("End:", df["Transaction Date"].max())

Start: 2011-01-01 00:00:00
End: 2017-12-09 00:00:00


In [8]:
n = len(df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = df.iloc[:train_end]
validation_df = df.iloc[train_end:validation_end]
test_df = df.iloc[validation_end:]

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (8112, 26)
Validation: (1738, 26)
Test: (1739, 26)


In [9]:
features = [
    "ATM Name",
    "Lag_1_Day",
    "Lag_7_Day",
    "Lag_14_Day",
    "Lag_30_Day",
    "Rolling_7_Day_Mean",
    "Rolling_14_Day_Mean",
    "Rolling_30_Day_Mean",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Is_Weekend",
    "Week_of_Year",
    "Working Day",
    "Holiday Sequence",
    "Festival Religion"
]

In [10]:
X_train = train_df[features]
y_train = train_df[target]

X_val = validation_df[features]
y_val = validation_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [11]:
categorical_features = [
    "ATM Name",
    "Festival Religion",
    "Working Day",
     "Holiday Sequence"
]

numerical_features = [
    "Lag_1_Day",
    "Lag_7_Day",
    "Lag_14_Day",
    "Lag_30_Day",
    "Rolling_7_Day_Mean",
    "Rolling_14_Day_Mean",
    "Rolling_30_Day_Mean",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Is_Weekend",
    "Week_of_Year",
   
]

In [12]:
print(df["Holiday Sequence"].unique())
print(df["Holiday Sequence"].dtype)

['WHH' 'WWW' 'WWH' 'HHW' 'HWW' 'HHH' 'WHW' 'HWH']
object


#preprocessing pipeline

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

#Baseline=> Tomorrow's demand ≈ previous day's demand.

In [14]:
baseline_predictions = X_test["Lag_1_Day"]

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_mape = mean_absolute_percentage_error(
    y_test,
    baseline_predictions
) * 100

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline MAPE:", baseline_mape)
print("Baseline R²:", baseline_r2)

Baseline MAE: 3584.0021161587124
Baseline RMSE: 5128.149523589184
Baseline MAPE: 1093.1472862112075
Baseline R²: -0.07619467892099152


In [15]:
print("Missing values in X_train:")
print(X_train.isnull().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum())

Missing values in X_train:
ATM Name               0
Lag_1_Day              0
Lag_7_Day              0
Lag_14_Day             0
Lag_30_Day             0
Rolling_7_Day_Mean     0
Rolling_14_Day_Mean    0
Rolling_30_Day_Mean    0
Year                   0
Month                  0
Day                    0
Day_of_Week            0
Is_Weekend             0
Week_of_Year           0
Working Day            0
Holiday Sequence       0
Festival Religion      0
dtype: int64

Missing values in X_test:
ATM Name               0
Lag_1_Day              0
Lag_7_Day              0
Lag_14_Day             0
Lag_30_Day             0
Rolling_7_Day_Mean     0
Rolling_14_Day_Mean    0
Rolling_30_Day_Mean    0
Year                   0
Month                  0
Day                    0
Day_of_Week            0
Is_Weekend             0
Week_of_Year           0
Working Day            0
Holiday Sequence       0
Festival Religion      0
dtype: int64


In [16]:
print("\nRows containing NaN in X_train:")
print(X_train[X_train.isnull().any(axis=1)].head())


Rows containing NaN in X_train:
Empty DataFrame
Columns: [ATM Name, Lag_1_Day, Lag_7_Day, Lag_14_Day, Lag_30_Day, Rolling_7_Day_Mean, Rolling_14_Day_Mean, Rolling_30_Day_Mean, Year, Month, Day, Day_of_Week, Is_Weekend, Week_of_Year, Working Day, Holiday Sequence, Festival Religion]
Index: []


#Linear regression

In [17]:
#Create
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)
#Train
linear_model.fit(
    X_train,
    y_train
)
#Predict
linear_predictions = linear_model.predict(X_test)
#Evaluate
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)
linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)
linear_mape = mean_absolute_percentage_error(
    y_test,
    linear_predictions
) * 100
linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("MAPE:", linear_mape)
print("R²:", linear_r2)

Linear Regression
MAE: 3062.617373595101
RMSE: 4098.479419499561
MAPE: 2493.0068525777133
R²: 0.31259134101471364


In [18]:
for col in features:
    if df[col].dtype == "object":
        print("\n", col)
        print(df[col].unique()[:20])


 ATM Name
['Airport ATM' 'Christ College ATM' 'Mount Road ATM' 'Big Street ATM'
 'KK Nagar ATM']

 Working Day
['H' 'W']

 Holiday Sequence
['WHH' 'WWW' 'WWH' 'HHW' 'HWW' 'HHH' 'WHW' 'HWH']

 Festival Religion
['C' 'H' 'NH' 'N' 'M']
